# API models (appendix) — run locally, no GPU

Runs the API models on the same four pair files as the local models (MedQA
and MedMCQA, race and gender; 1,000 pairs each × 3 prompts). The GPU notebook (`notebooks/colab_local_models.ipynb`) runs the local models; both
write to separate model folders, and the Colab analysis (sections 6–8) merges
them.

| Model | Confidence |
|---|---|
| gpt-5.6-terra | TLP if the API returns logprobs, else CE |
| gpt-4.1-mini | TLP + CE (the GPT that still returns logprobs) |
| claude-sonnet-5 | CE (Anthropic returns no logprobs) |

**Merging:** if Google Drive for desktop is installed, results go straight to
`My Drive/CARE/results` and there is nothing to do. Otherwise the last cell packs
them into `api_results.zip`; upload it to `My Drive/CARE/` and the Colab setup
cell unpacks it.

API keys are asked for once and never written to this notebook. Every call is
cached as soon as it returns, so an interrupted run resumes without paying
twice. On a laptop, run `caffeinate -i` in a terminal so the Mac does not sleep.

## 1. Setup

In [ ]:
import os, sys, json, glob, hashlib
from pathlib import Path
CARE = Path.cwd() if (Path.cwd() / "care").is_dir() else Path.cwd().parent
sys.path.insert(0, str(CARE)); os.chdir(CARE)

# Results straight into Google Drive for desktop when installed, else locally.
drive = glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive-*/My Drive/CARE"))
os.environ.setdefault("CARE_DATA_ROOT", f"{drive[0]}/results" if drive else str(CARE / "results_api"))
os.environ.setdefault("CARE_OUTPUT_ROOT", str(CARE / "outputs_api"))

from getpass import getpass
for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    if not os.environ.get(k):
        os.environ[k] = getpass(f"{k} (kept in memory only): ")

import pandas as pd
from care import config as cfg, infer                    # after the env vars
from care.prompts import SYSTEM, CONDITIONS
from care.data import load_records, validate
from care import analysis

RESULTS = Path(cfg.DATA_ROOT)
print("results ->", RESULTS, "(Google Drive)" if drive else "(local; upload at the end)")
print("code version:", getattr(infer, "CODE_VERSION", "OLD -- restart the kernel"))

In [ ]:
N_OPTIONS = {"pairs_medqa_race": 5, "pairs_medqa_gender": 5,
             "pairs_medmcqa_race": 4, "pairs_medmcqa_gender": 4}
# API models are the appendix and run on the primary dataset only. Add more
# pair files here to extend them (6,000 calls per model per method each).
DATASETS = ["pairs_medqa_race"]
PAIR_FILES = {d: Path(cfg.PAIRS_DIR) / f"{d}.jsonl" for d in DATASETS}
if not all(p.exists() for p in PAIR_FILES.values()):         # build once from the source corpora
    import subprocess
    subprocess.run([sys.executable, "-m", "care.pairgen"], check=True)
PAIRS = {d: [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
         for d, f in PAIR_FILES.items()}
for d, r in PAIRS.items():
    print(f"{d:22s} {len(r)//2} pairs, {len(r)} items x {len(CONDITIONS)} prompts")

# Must match the GPU run: compare with run_manifest.json from Colab.
sha = lambda b: hashlib.sha256(b).hexdigest()[:12]
for d, f in PAIR_FILES.items():
    print(f"{d} sha256:", sha(f.read_bytes()))
for c in CONDITIONS:
    print(f"prompt {c} sha256:", sha(SYSTEM[c].encode()))

## 2. Models

Check the ids against what your keys can see.

In [ ]:
from openai import OpenAI
import anthropic
try:
    print("OpenAI:",    sorted(m.id for m in OpenAI().models.list() if m.id.startswith("gpt")))
except Exception as e:
    print("OpenAI: cannot list models --", type(e).__name__, str(e)[:150])
try:
    print("Anthropic:", sorted(m.id for m in anthropic.Anthropic().models.list().data))
except Exception as e:
    print("Anthropic: cannot list models --", type(e).__name__, str(e)[:150])

In [ ]:
APPENDIX = {
    "gpt-6-sol":               "openai",     # newest GPT
    "gpt-5.6-terra":           "openai",     # already complete on MedQA race
    "gpt-4.1-mini-2025-04-14": "openai",     # returns logprobs -> TLP
    "claude-sonnet-5":         "anthropic",  # no logprobs -> CE
}
# Per-model override; empty means every model runs every dataset in DATASETS.
DATASETS_FOR = {}
ds_for = lambda m: [d for d in DATASETS_FOR.get(m, DATASETS) if d in DATASETS]

METHODS = {}
for model, backend in APPENDIX.items():
    ok, msg = infer.APIModel(backend, model).supports_logprobs()
    METHODS[model] = ("tlp", "ce") if ok else ("ce",)
    print(f"{model:26s} TLP {'yes' if ok else 'no '} -> {METHODS[model]}  on "
          f"{[d.replace('pairs_', '') for d in ds_for(model)]}   ({msg})")

### Optional: start a model over

Leave `CLEAR` empty to resume. Put a model in it to delete its stored results
before the pilot (permanent). Records from an older CE instruction are
ignored and re-run automatically anyway, so this is rarely needed.

In [ ]:
import shutil
CLEAR = []        # e.g. ["claude-sonnet-5"]; keep [] to resume
for model in CLEAR:
    for d in ds_for(model):
        p = RESULTS / d / model
        if p.exists():
            shutil.rmtree(p); print("deleted", p)

## 3. Pilot: 20 pairs first

Check the parse rate and a raw reply before paying for the full run. The pilot
items are part of the full run, so nothing is spent twice.

In [ ]:
def subset(n_pairs):
    out = {}
    for d, recs in PAIRS.items():
        keep = set(list(dict.fromkeys(r["pair_id"] for r in recs))[:n_pairs])
        out[d] = [r for r in recs if r["pair_id"] in keep]
    return out

def run(pairs, workers=8):
    for model, backend in APPENDIX.items():
        print(f"\n=== {model} ===")
        api = infer.APIModel(backend, model)
        try:
            mine = {d: r for d, r in pairs.items() if d in ds_for(model)}
            infer.run_api(api, model, mine, N_OPTIONS, SYSTEM, CONDITIONS,
                          METHODS[model], RESULTS, max_workers=workers)
        except RuntimeError as e:        # circuit breaker: calls keep failing
            print("  STOPPED:", str(e)[:300])
        print("  parameters used:", api.params, "| resolved model:", api.resolved_model)

def check(items):
    """Per dataset x model x method x prompt; flags anything that would waste the full run."""
    rows, flags = [], []
    for d, recs in items.items():
        want = {(str(r["question_id"]), str(r["variant"]).lower()) for r in recs}
        chance = 1 / N_OPTIONS[d]
        for model in APPENDIX:
            if d not in ds_for(model):
                continue
            for meth in METHODS[model]:
                try:
                    df = load_records(d, model, meth)
                except FileNotFoundError:
                    flags.append(f"{d} {model} {meth}: no results"); continue
                df = df[[(q, v) in want for q, v in zip(df.question_id, df.variant)]]
                for c in CONDITIONS:
                    s = df[df.prompt == c]
                    r = {"dataset": d.replace("pairs_", ""), "model": cfg.short(model), "method": meth,
                         "prompt": c, "n": len(s), "parse": s.parse_ok.mean(), "acc": s.correct.mean(),
                         "mean_conf": s.conf_raw.mean(), "distinct": s.conf_raw.nunique(),
                         "letter_mass": s.letter_mass.median()}
                    rows.append(r)
                    tag = f"{r['dataset']} {r['model']} {meth} {c}"
                    if len(s) < len(recs):
                        flags.append(f"{tag}: only {len(s)}/{len(recs)} items done (re-run this cell)")
                    if len(s) and r["parse"] < 0.9:
                        flags.append(f"{tag}: parse rate {r['parse']:.2f}")
                    if len(s) and r["acc"] < chance + 0.1:
                        flags.append(f"{tag}: accuracy {r['acc']:.2f} near chance {chance:.2f}")
                    if meth == "tlp" and len(s) and r["letter_mass"] < 0.5:
                        flags.append(f"{tag}: letter mass {r['letter_mass']:.2f}")
    with pd.option_context("display.max_rows", 500):
        print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.2f}"))
    print("\nFLAGS:" if flags else "\nno flags -- go ahead with the full run")
    for f in flags:
        print("  ", f)
    # unparsed CE replies under the current instruction, re-parsed with the current parser
    for model in APPENDIX:
        if "ce" not in METHODS[model]:
            continue
        bad, n = [], 0
        for d, recs in items.items():
            if d not in ds_for(model):
                continue
            want = {(str(r["question_id"]), str(r["variant"]).lower()) for r in recs}
            for c in CONDITIONS:
                path = infer.out_path(RESULTS, d, model, c)
                latest = {}                      # files are append-only: keep the newest reply
                for l in (open(path) if path.exists() else []):
                    r = json.loads(l)
                    key = (str(r["question_id"]), str(r["variant"]).lower())
                    if key in want and r.get("ce_version") == infer.CE_VERSION and (r.get("ce_raw_text") or "").strip():
                        latest[key] = r
                n += len(latest)
                bad += [r["ce_raw_text"] for r in latest.values()
                        if infer.parse_ce(r["ce_raw_text"], infer.LETTERS[:N_OPTIONS[d]])[0] is None]
        print(f"\n{cfg.short(model)}: {len(bad)}/{n} CE replies unparsed")
        for b in bad[:3]:
            print("   ", repr(b[:120]))

PILOT_ITEMS = subset(20)
run(PILOT_ITEMS)
check(PILOT_ITEMS)

Go on only if it prints **no flags -- go ahead with the full run**. The pilot items count towards the full run.

## 4. Full run

In [ ]:
run(PAIRS)
check(PAIRS)

## 5. Quick look, manifest, hand-over

In [ ]:
for d in DATASETS:
    for model in APPENDIX:
        if d not in ds_for(model):
            continue
        for meth in METHODS[model]:
            try:
                rep = validate(load_records(d, model, meth), strict=False)
                print(f"{d}/{model}/{meth}: errors={rep.errors or '-'}; warnings={len(rep.warnings)}")
            except FileNotFoundError:
                print(f"{d}/{model}/{meth}: not run")

appx = analysis.run(datasets=DATASETS, models=list(APPENDIX), methods=("tlp", "ce"),
               verbose=False, tag="appendix_local")
if len(appx["sar"]):
    cols = ["dataset", "model", "method", "prompt", "acc_ref", "acc_dem", "sar_ref", "sar_dem",
            "sar_gap", "sar_gap_lo", "sar_gap_hi", "ceiling"]
    for al in (0.10, 0.05):
        a = appx["sar"][appx["sar"].alpha == al]
        print(f"\n=== SAR, alpha = {al} (ceiling = both groups at SAR 1: gap uninformative) ===")
        print(a[cols].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
        ac = appx["contrast"][appx["contrast"].alpha == al]
        print(f"\n--- P2/P3 vs P1, alpha = {al}: change in gap and in average SAR ---")
        print(ac[["dataset", "model", "method", "condition", "d_gap", "d_gap_lo", "d_gap_hi", "d_sar_mean",
                  "d_sar_mean_lo", "d_sar_mean_hi", "costs_automation", "ceiling"]]
              .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

META = {m: {"backend": b, "dir": m, "methods": list(METHODS[m])} for m, b in APPENDIX.items()}
print(infer.write_manifest(RESULTS / "api_manifest.json",
                           {d: str(f) for d, f in PAIR_FILES.items()}, SYSTEM, META))

In [ ]:
# Pack the API results for Colab (skip if they already live on Google Drive).
import zipfile
if drive:
    print("already on Google Drive -- nothing to upload")
else:
    out = CARE / "api_results.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for d in DATASETS:
            for model in APPENDIX:
                if d not in ds_for(model):
                    continue
                for f in (RESULTS / d / model).rglob("*.jsonl"):
                    z.write(f, f.relative_to(RESULTS))
        z.write(RESULTS / "api_manifest.json", "api_manifest.json")
    print(f"upload {out} to My Drive/CARE/ -- the Colab setup cell unpacks it into results/")